In [6]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. SETTINGS: change these paths and column names
# --------------------------------------------------
input_dir = Path(r"D:\B12")
output_csv = "data/BIO12_temporal_standard_deviation.csv"
output_gpkg = "data/BIO12_temporal_standard_deviation.gpkg"

grid_id_col = "grid"       # Unique grid-cell identifier
mean_col = "_mean"     # Change to "mean" if that is your field

# Use population SD because the 539 time slices are
# treated as the complete set of periods being analysed.
ddof = 0

# --------------------------------------------------
# 2. FIND ALL GEOPACKAGE FILES
# --------------------------------------------------
gpkg_files = sorted(input_dir.glob("*.gpkg"))

if not gpkg_files:
    raise FileNotFoundError("No .gpkg files found in the input folder.")

# --------------------------------------------------
# 3. READ THE GRID GEOMETRY AND MEAN BIO1 VALUES
# --------------------------------------------------
bio12_tables = []
reference_grid = None

for i, file in enumerate(gpkg_files, start=1):

    gdf = gpd.read_file(file)

    # Keep the first GeoPackage as the reference geometry
    if reference_grid is None:
        if grid_id_col not in gdf.columns:
            raise ValueError(f"Grid ID '{grid_id_col}' not found in {file.name}. " f"Available columns: {list(gdf.columns)}")

        reference_grid = gdf[[grid_id_col, "geometry"]].copy()

    # Check required columns
    missing_cols = [c for c in [grid_id_col, mean_col] if c not in gdf.columns]

    if missing_cols:
        raise ValueError(f"Missing columns {missing_cols} in {file.name}. "f"Available columns: {list(gdf.columns)}")

    # Keep only the grid ID and mean BIO1
    df = gdf[[grid_id_col, mean_col]].copy()

    # Convert mean values to numeric
    df[mean_col] = pd.to_numeric(df[mean_col], errors="coerce" )

    # Ensure every grid appears only once per time slice
    if df[grid_id_col].duplicated().any():
        raise ValueError(f"Duplicate grid IDs found in {file.name}")

    # Name the column after the time-slice file
    time_name = file.stem
    df = df.rename(columns={mean_col: time_name})

    bio12_tables.append(df)

    if i % 50 == 0 or i == len(gpkg_files):
        print(f"Read {i}/{len(gpkg_files)} files")

# --------------------------------------------------
# 4. COMBINE ALL TIME SLICES BY GRID ID
# --------------------------------------------------
bio12_all = bio12_tables[0]

for df in bio12_tables[1:]:
    bio12_all = bio12_all.merge(df,on=grid_id_col,how="outer",validate="one_to_one")

time_cols = [c for c in bio12_all.columns if c != grid_id_col]

print("Number of time slices:", len(time_cols))
print("Number of grid cells:", len(bio12_all))

# --------------------------------------------------
# 5. CALCULATE TEMPORAL STANDARD DEVIATION
# --------------------------------------------------
values = bio12_all[time_cols]

bio12_all["BIO12_n_valid"] = values.count(axis=1)

bio12_all["BIO12_temporal_SD"] = values.std(axis=1,ddof=ddof,skipna=True)

# Optional: temporal mean, useful for quality checking
bio12_all["BIO12_temporal_mean"] = values.mean(axis=1,skipna=True)

# --------------------------------------------------
# 6. SAVE RESULTS AS CSV
# --------------------------------------------------
summary = bio12_all[[grid_id_col,"BIO12_temporal_mean","BIO12_temporal_SD","BIO12_n_valid"]].copy()

summary.to_csv(output_csv, index=False)

# --------------------------------------------------
# 7. SAVE RESULTS AS GEOPACKAGE FOR QGIS
# --------------------------------------------------
result_gdf = reference_grid.merge(summary,on=grid_id_col,how="left",validate="one_to_one")

result_gdf.to_file(output_gpkg,layer="BIO12_temporal_SD",driver="GPKG")

print("\nCalculation completed!")
print("CSV:", output_csv)
print("GeoPackage:", output_gpkg)
print("\nSummary of temporal SD:")
print(result_gdf["BIO12_temporal_SD"].describe())

print("\nGrid cells with fewer than 539 valid values:")
print((result_gdf["BIO12_n_valid"] < len(gpkg_files)).sum())


Read 100/538 files
Read 200/538 files
Read 300/538 files
Read 350/538 files
Read 400/538 files
Read 450/538 files
Read 500/538 files
Read 538/538 files
Number of time slices: 538
Number of grid cells: 114947


C:\Users\vainq\AppData\Local\Temp\ipykernel_5936\1643339944.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bio12_all["BIO12_n_valid"] = values.count(axis=1)
C:\Users\vainq\AppData\Local\Temp\ipykernel_5936\1643339944.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bio12_all["BIO12_temporal_SD"] = values.std(axis=1,ddof=ddof,skipna=True)
C:\Users\vainq\AppData\Local\Temp\ipykernel_5936\1643339944.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,


Calculation completed!
CSV: data/BIO12_temporal_standard_deviation.csv
GeoPackage: data/BIO12_temporal_standard_deviation.gpkg

Summary of temporal SD:
count    114931.000000
mean         60.899256
std          33.440029
min           0.000000
25%          36.984954
50%          57.412588
75%          82.773524
max         277.403649
Name: BIO12_temporal_SD, dtype: float64

Grid cells with fewer than 539 valid values:
764


In [5]:
gdf = gpd.read_file('D:/B12/tbio12_t60.gpkg', driver="GPKG")

C:\Users\vainq\anaconda3\envs\UICNco-occurence_map_analysis\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


In [13]:
gdf

,grid,weight_list,WE,richness,CWE,_count,_sum,_mean,geometry
0,308,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.083605,509,0.000164,0.126841,34.119461,268.994726,"MULTIPOLYGON (((7.97286 5.20776, 7.96912 5.212..."
1,309,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.083577,509,0.000164,0.024023,6.473439,269.470379,"MULTIPOLYGON (((7.97019 5.25216, 7.9713 5.2538..."
2,318,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.079305,509,0.000156,0.031175,8.510836,273.000000,"MULTIPOLYGON (((7.97236 5.73213, 7.96792 5.745..."
3,319,"[6.941070313042272e-05, 3.8892345986309894e-05...",0.079407,514,0.000154,0.549717,150.109159,273.066154,"MULTIPOLYGON (((7.96499 5.75398, 7.95362 5.772..."
4,320,"[6.941070313042272e-05, 3.417751802864076e-05,...",0.079317,507,0.000156,1.172320,320.657990,273.524177,"MULTIPOLYGON (((7.93881 5.79709, 7.93473 5.805..."
...,...,...,...,...,...,...,...,...,...
114942,177138,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.236881,666,0.000356,0.014225,3.257566,229.000000,"MULTIPOLYGON (((32.05357 1.99546, 32.06525 1.9..."
114943,177503,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.185886,659,0.000282,0.000038,0.008628,229.000000,"MULTIPOLYGON (((32.07396 1.94478, 32.07396 1.9..."
114944,177504,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.243488,666,0.000366,0.562589,128.832768,229.000000,"MULTIPOLYGON (((32.12164 1.99455, 32.10951 1.9..."
114945,177505,"[6.157635467980295e-05, 9.470414425335253e-06,...",0.243263,665,0.000366,0.408008,93.433908,229.000000,"MULTIPOLYGON (((32.07656 1.99885, 32.09024 2.0..."
